# Phase 2 - Legal NER (RNN + Transformer **fusion** Phase-1 model)

**Task:** predict character spans `[start, end, label]` for 12 legal entity types; scored by *strict* macro-F1 (exact start, end and label).

**What you need on disk (nothing else - the model code is embedded below, no .py files):**
1. your Phase-1 checkpoint `.pt` (`fusion.pt`) - it must contain `model_state` and `config` (the checkpoint written by the Phase-1 fusion notebook does)
2. the Phase-1 tokenizer pickle `legal_bpe_v3.pkl` - **the `.pt` does not contain the tokenizer**, and you must use the same one the model was trained with
3. a Hugging Face token (Kaggle secret `CS781_WRITE`) to download the Phase-2 data

**Approach:** BIO token tagging with a **BiGRU + linear head** on top of the fusion backbone. The fusion LM is strictly causal in *both* of its paths (masked attention and a unidirectional GRU), so a token still cannot see its right-hand context; the BiGRU head gives it back. Long documents are handled with overlapping 512-token windows; predictions are averaged over overlaps.

**Experiments (4 submissions):** full fine-tuning, LoRA, partial (layer-wise) fine-tuning, knowledge distillation.

### What changed relative to `phase-2-decoder.ipynb`

| Part | Decoder notebook | This notebook |
|---|---|---|
| Embedded model code | `CausalSelfAttention`, `DecoderLM` (sinusoidal pos-emb, LayerNorm, SwiGLU) | `FusionConfig`, `FusionLM` (RoPE, RMSNorm, GELU MLP, **parallel attention + GRU with a learned gate**), copied unchanged from Phase 1 |
| Checkpoint loading | `LMConfig`, `DecoderLM` | `FusionConfig`, `FusionLM` |
| `NERModel.features` | token emb + `pos_emb`, `block(h)` | token emb, then `block(h, cos, sin)` with RoPE tables, `norm_f` (no additive position embedding) |
| LoRA | all `nn.Linear` in blocks | same, which now also covers `qkv`, the gate, and the GRU output projection `rnn.proj`; the GRU's own weights stay frozen |
| BitFit | bias terms | this model has almost no biases, so it is redefined as biases + norm scales |
| KD student | keep every 2nd block | keep alternate *pairs* of blocks, so the RNN / no-RNN pattern of the teacher is preserved (every 2nd block would give an all-GRU student that cannot load the teacher's weights cleanly) |
| Experiment 2 | fit on fold1+fold2 with no validation | validated like the others (fold1 -> fold2), so the summary table is comparable; the fold1+fold2 retrain is Section 14 |

> **Not covered:** the course document calls **LORMA compulsory** for Phase 2. It is not in this notebook. The `LoRALinear` below is plain LoRA, a starting point for your own LORMA implementation from the paper (Bihany, Patel, Modi 2025). Check with the TAs which 4 experiments they expect.
>
> Learning rates / epochs below are only starting values carried over from the decoder run - choosing and justifying them for the fusion model is your job (course rules).

## 0. Config - edit the paths

In [ ]:
TOKENIZER_PATH = "/kaggle/input/models/harsvij/cap-tok/pytorch/default/1/legal_bpe_v3.pkl"      # <-- Phase-1 tokenizer
CKPT_PATH      = "/kaggle/input/models/harsvij/phase-1-models/pytorch/default/1/fusion.pt"      # <-- Phase-1 fusion checkpoint (check this path)
HF_REPO = "Exploration-Lab/CS-781-Capstone"
OUT = "/kaggle/working"                                              # where zips / checkpoints are written
WIN, STRIDE = 512, 256                                               # window / stride in tokens (the LM was pre-trained at 1024, so up to WIN=1024 is allowed)

## 1. Imports and Hugging Face login

In [ ]:
import copy, json, math, os, pickle, random, re, time, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| torch", torch.__version__)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("CS781_WRITE")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")        # Colab / local: export HF_TOKEN yourself
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN); print("HF login ok")
else:
    print("no HF token found - dataset download may fail")

## 2. Phase-1 tokenizer (class copied from your Phase-1 notebook)

In [ ]:
class LegalBPETokenizer:
    """Phase-1 tokenizer, copied unchanged from your Phase-1 notebook (needs legal_bpe_v3.pkl)."""

    def __init__(self, model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])
        self.merge_rank = {tuple(pair): rank for rank, (pair, _) in enumerate(self.merges)}
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]
        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")
        self._special_re = (re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
                            if self.special_tokens else None)
        self._cache = {}

    def _encode_piece(self, piece_bytes):
        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached
        ids = list(piece_bytes)
        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break
            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):
        if self.lowercase:
            text = text.lower()
        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):
        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])
        return b"".join(chunks).decode("utf-8", errors="replace")


tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
print("vocab size:", tokenizer.vocab_size, "| lowercase:", tokenizer.lowercase, "| eos id:", tokenizer.eos_id)
print([tokenizer.decode([i]) for i in tokenizer.encode("The appellant filed this appeal.")])

## 3. Phase-1 fusion model code (copied unchanged from your Phase-1 notebook) and checkpoint loading
Embedding the class definitions is what lets the notebook work with only the `.pt` file. This is the `model_arch_fusion.py` from Phase 1, pasted as-is: every name and parameter shape must match the checkpoint, so do not edit it.

In [ ]:
# ---- model_arch_fusion.py (Phase 1, unchanged) ----
from dataclasses import dataclass


@dataclass
class FusionConfig:

    vocab_size: int

    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2816

    max_seq_len: int = 1024

    dropout: float = 0.0

    # ---- fusion-specific fields ----
    rnn_hidden: int = 512         # GRU hidden size (projected back to d_model)
    rnn_every: int = 2            # GRU branch in blocks where i % rnn_every == 0; 0 = no RNN
    rnn_fp32: bool = True         # run the recurrent branch in fp32 even under fp16 autocast
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5


class RMSNorm(nn.Module):

    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (out * self.weight.float()).type_as(x)


def build_rope(head_dim: int, max_len: int, theta: float):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)                 # (L, hd/2)
    emb = torch.cat([freqs, freqs], dim=-1)          # (L, hd)
    return emb.cos(), emb.sin()


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # x: (B, H, T, hd); cos/sin: (T, hd). Computed in fp32, returned in x.dtype
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).type_as(x)


class CausalSelfAttention(nn.Module):

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        assert self.head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)                                   # each (B, H, T, hd)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.out(y)


class RecurrentBranch(nn.Module):
    """Unidirectional GRU (strictly causal) followed by a projection to d_model."""

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        self.fp32 = cfg.rnn_fp32
        self.gru = nn.GRU(cfg.d_model, cfg.rnn_hidden, num_layers=1,
                          batch_first=True, bidirectional=False)
        self.proj = nn.Linear(cfg.rnn_hidden, cfg.d_model, bias=False)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        out_dtype = h.dtype
        if self.fp32:
            # keep the sequential recurrence out of fp16
            with torch.autocast(device_type=h.device.type, enabled=False):
                y, _ = self.gru(h.float())
        else:
            y, _ = self.gru(h)
        return self.proj(y.to(out_dtype))


class MLP(nn.Module):

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class FusionBlock(nn.Module):

    def __init__(self, cfg: FusionConfig, use_rnn: bool):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = CausalSelfAttention(cfg)
        self.rnn = RecurrentBranch(cfg) if use_rnn else None
        self.gate = nn.Linear(cfg.d_model, cfg.d_model, bias=True) if use_rnn else None
        self.norm2 = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        h = self.norm1(x)
        mixed = self.attn(h, cos, sin)
        if self.rnn is not None:
            r = self.rnn(h)
            g = torch.sigmoid(self.gate(h))
            mixed = (1.0 - g) * mixed + g * r
        x = x + self.drop(mixed)
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class FusionLM(nn.Module):
    """forward(idx: LongTensor (B, T)) -> logits (B, T, vocab_size); strictly causal; 1 <= T <= cfg.max_seq_len."""

    def __init__(self, cfg: FusionConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([
            FusionBlock(cfg, use_rnn=(cfg.rnn_every > 0 and i % cfg.rnn_every == 0))
            for i in range(cfg.n_layers)
        ])
        self.norm_f = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight             # weight tying

        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # GPT-2 style: scale the projections that write into the residual stream
        res_std = 0.02 / math.sqrt(2 * cfg.n_layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "mlp.fc2.weight", "rnn.proj.weight")):
                nn.init.normal_(p, mean=0.0, std=res_std)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        assert 1 <= T <= self.cfg.max_seq_len, f"sequence length {T} outside [1, {self.cfg.max_seq_len}]"
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        for blk in self.blocks:
            x = blk(x, cos, sin)
        x = self.norm_f(x)
        return self.lm_head(x)

In [ ]:
def load_lm(path=CKPT_PATH):
    # weights_only=False: this is YOUR OWN checkpoint (config dict + tensors), not an untrusted file
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = FusionConfig(**{k: v for k, v in ck["config"].items() if k in FusionConfig.__dataclass_fields__})
    assert cfg.vocab_size == tokenizer.vocab_size, \
        f"checkpoint vocab {cfg.vocab_size} != tokenizer vocab {tokenizer.vocab_size}: wrong tokenizer .pkl?"
    lm = FusionLM(cfg)
    state = {k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k: v     # in case it was saved from torch.compile
             for k, v in ck["model_state"].items()}
    lm.load_state_dict(state)
    return lm

In [ ]:
_lm = load_lm()
print(_lm.cfg)
n_rnn = sum(b.rnn is not None for b in _lm.blocks)
print(f"loaded Phase-1 fusion LM: {sum(p.numel() for p in _lm.parameters())/1e6:.1f}M params | "
      f"{len(_lm.blocks)} blocks, GRU branch in {n_rnn} of them")
assert WIN <= _lm.cfg.max_seq_len, f"WIN={WIN} exceeds the LM's max_seq_len={_lm.cfg.max_seq_len} (RoPE tables are that long)"

## 4. Data
### 4.1 Peek at the raw JSON (verify the layout before trusting the parsers)

In [ ]:
# Look at the raw JSON layout BEFORE trusting the parsers below (the exact schema was not visible to me).
def hf_json(path):
    return json.load(open(hf_hub_download(HF_REPO, path, repo_type="dataset"), encoding="utf-8"))

raw_text1 = hf_json("Phase_2_LNER/Train/fold1-text.json")
raw_gold1 = hf_json("Phase_2_LNER/Train/fold1-gold.json")
for name, raw in [("fold1-text", raw_text1), ("fold1-gold", raw_gold1)]:
    print(name, type(raw).__name__, "len", len(raw))
    first = next(iter(raw.items())) if isinstance(raw, dict) else raw[0]
    print("   first item:", str(first)[:400], "\n")
print(open(hf_hub_download(HF_REPO, "Phase_2_LNER/Train/labels.txt", repo_type="dataset")).read())

### 4.2 Parse folds, labels - and check that gold offsets slice out real entity text

In [ ]:
def norm_text(t):
    if isinstance(t, list):
        t = {g["id"]: g for g in t}
    return {str(k): (v["text"] if isinstance(v, dict) else v) for k, v in t.items()}

def norm_gold(gold):
    """-> {doc_id: [(start, end, label), ...]}; tolerant to dict / list layouts."""
    items = gold.items() if isinstance(gold, dict) else \
        ((g["id"], g.get("entities", g.get("annotations", []))) for g in gold)
    out = {}
    for did, ents in items:
        lst = []
        for e in ents:
            if isinstance(e, dict):
                s = e.get("start", e.get("start_char")); t = e.get("end", e.get("end_char")); l = e.get("label")
            else:
                s, t, l = e[0], e[1], e[2]
            lst.append((int(s), int(t), l))
        out[str(did)] = lst
    return out

def load_fold(k):
    return norm_text(hf_json(f"Phase_2_LNER/Train/fold{k}-text.json")), \
           norm_gold(hf_json(f"Phase_2_LNER/Train/fold{k}-gold.json"))

labels = [l.strip() for l in open(hf_hub_download(HF_REPO, "Phase_2_LNER/Train/labels.txt",
                                                  repo_type="dataset"), encoding="utf-8") if l.strip()]
label2k = {l: i for i, l in enumerate(labels)}
N_TAGS = 1 + 2 * len(labels)                       # O + (B, I) per label
print(len(labels), "labels:", labels)

t1, g1 = load_fold(1);  t2, g2 = load_fold(2)
print("fold1:", len(t1), "docs |", sum(map(len, g1.values())), "entities")
print("fold2:", len(t2), "docs |", sum(map(len, g2.values())), "entities")

# THE offset sanity check: these slices must look like the entity names/labels.
d0 = next(iter(t1))
print([(s, e, l, t1[d0][s:e]) for s, e, l in g1[d0][:6]])
unknown = {l for g in (g1, g2) for es in g.values() for _, _, l in es} - set(labels)
print("labels in gold but not in labels.txt:", unknown)

### 4.3 Tokenisation with character offsets, BIO tags, span decoding

In [ ]:
def _safe_lower(text):
    # lower-case without ever changing the string length, so character offsets stay valid
    return "".join(c.lower() if len(c.lower()) == 1 else c for c in text)

def encode_with_offsets(text):
    """BPE-encode `text` and return (ids, offsets) where offsets[i] = (char_start, char_end) of token i."""
    t = _safe_lower(text) if tokenizer.lowercase else text
    ids, offs = [], []
    for m in tokenizer.pattern.finditer(t):
        piece = m.group(0)
        if not piece:
            continue
        b2c = []                                    # byte index inside the piece -> char index
        for ci, ch in enumerate(piece):
            b2c.extend([ci] * len(ch.encode("utf-8")))
        pos = 0
        for tid in tokenizer._encode_piece(piece.encode("utf-8")):
            n = len(tokenizer.token_bytes[tid])
            ids.append(tid)
            offs.append((m.start() + b2c[pos], m.start() + b2c[pos + n - 1] + 1))
            pos += n
    return ids, offs

# tag ids: 0 = O, 1+2k = B-label_k, 2+2k = I-label_k
def make_tags(offs, ents, label2k):
    tags = [0] * len(offs)
    for s, e, l in sorted(ents, key=lambda x: -(x[1] - x[0])):      # longer spans first
        idx = [i for i, (a, b) in enumerate(offs) if a < e and b > s and tags[i] == 0]
        if not idx:
            continue
        k = label2k[l]
        tags[idx[0]] = 1 + 2 * k
        for i in idx[1:]:
            tags[i] = 2 + 2 * k
    return tags

def decode(tag_ids, offs, text, labels):
    """Token tags -> [(char_start, char_end, label)], with leading/trailing whitespace stripped."""
    ents, cur = [], None
    def close():
        nonlocal cur
        if cur is not None:
            s, e, k = cur
            while s < e and text[s].isspace(): s += 1          # BPE tokens carry their leading space
            while e > s and text[e - 1].isspace(): e -= 1
            if e > s:
                ents.append((s, e, labels[k]))
        cur = None
    for i, t in enumerate(tag_ids):
        if t == 0:
            close(); continue
        k, is_b = (t - 1) // 2, (t - 1) % 2 == 0
        if is_b or cur is None or cur[2] != k:
            close(); cur = [offs[i][0], offs[i][1], k]
        else:
            cur[1] = offs[i][1]
    close()
    return ents

def build_docs(text, gold, label2k):
    docs = []
    for did, t in text.items():
        ids, offs = encode_with_offsets(t)
        ents = gold.get(did, []) if gold is not None else []
        docs.append(dict(id=did, text=t, ids=ids, offs=offs, ents=ents,
                         tags=make_tags(offs, ents, label2k) if gold is not None else None))
    return docs

### 4.4 Strict metric (exact start, end, label) - local re-implementation

In [ ]:
def strict_scores(pred, gold, labels):
    """Strict (exact start, end, label) per-label P/R/F1, macro-averaged over all labels."""
    P, R, F1 = [], [], []
    for lab in labels:
        pset = {(d, s, e) for d, es in pred.items() for s, e, l in es if l == lab}
        gset = {(d, s, e) for d, es in gold.items() for s, e, l in es if l == lab}
        tp = len(pset & gset)
        p = tp / len(pset) if pset else 0.0
        r = tp / len(gset) if gset else 0.0
        P.append(p); R.append(r); F1.append(2 * p * r / (p + r) if p + r else 0.0)
    return dict(macro_f1=float(np.mean(F1)), macro_p=float(np.mean(P)), macro_r=float(np.mean(R)),
                per_label=dict(zip(labels, F1)))

def alignment_check(docs, labels):
    """Ceiling check: gold -> token tags -> decoded spans. Must be close to 1.0 BEFORE you train anything."""
    pred = {d["id"]: decode(d["tags"], d["offs"], d["text"], labels) for d in docs}
    gold = {d["id"]: d["ents"] for d in docs}
    sc = strict_scores(pred, gold, labels)
    print(f"alignment ceiling macro-F1 = {sc['macro_f1']:.4f}")
    return sc

### 4.5 Build documents + alignment ceiling
The ceiling check converts gold spans to token tags and back. If it is well below 1.0, fix tokenisation/offsets before spending GPU time. (Tokenisation is identical to the decoder notebook, because the tokenizer is the same.)

In [ ]:
t0 = time.time()
docs1, docs2 = build_docs(t1, g1, label2k), build_docs(t2, g2, label2k)
test_docs = build_docs(norm_text(hf_json("Phase_2_LNER/Test/fold3-text.json")), None, label2k)
print(f"tokenised in {time.time()-t0:.0f}s | tokens/doc (fold1): "
      f"median {int(np.median([len(d['ids']) for d in docs1]))}, max {max(len(d['ids']) for d in docs1)}")
print("test docs:", len(test_docs))
ceil1 = alignment_check(docs1, labels)
print({k: round(v, 3) for k, v in ceil1["per_label"].items()})   # a label far below 1.0 = boundary problem for it

### 4.6 Checkpoint / tokenizer sanity check

In [ ]:
# Does checkpoint + tokenizer agree? The LM loss on real legal text should be far below ln(V) ~ 10.6
# (a few nats; compare with the val loss at the end of your Phase-1 run). ~10 means a tokenizer / checkpoint mismatch.
_lm.eval()
d = max(docs1, key=lambda d: len(d["ids"]))           # longest fold1 document
x = torch.tensor([d["ids"][:min(512, _lm.cfg.max_seq_len)]])
with torch.no_grad():
    lg = _lm(x)
print("Phase-1 LM loss on a fold1 snippet:", round(F.cross_entropy(lg[0, :-1], x[0, 1:]).item(), 3),
      "| ln(V) =", round(math.log(tokenizer.vocab_size), 2))
del _lm

### 4.7 Windows and batching

In [ ]:
def win_starts(n, win, stride):
    st = list(range(0, max(n - win, 0) + 1, stride))
    if st[-1] + win < n:
        st.append(n - win)
    return st

def make_windows(docs, win=WIN, stride=STRIDE):
    out = []
    for d in docs:
        if not d["ids"]:
            continue
        for s in win_starts(len(d["ids"]), win, stride):
            out.append((d["ids"][s:s + win], d["tags"][s:s + win]))
    return out

In [ ]:
def collate(batch):
    L = max(len(x) for x, _ in batch)
    X = torch.zeros(len(batch), L, dtype=torch.long)
    Y = torch.full((len(batch), L), -100, dtype=torch.long)
    lens = torch.tensor([len(x) for x, _ in batch])
    for i, (x, y) in enumerate(batch):
        X[i, :len(x)] = torch.tensor(x)
        if y is not None:
            Y[i, :len(y)] = torch.tensor(y)
    return X, Y, lens

In [ ]:
def iter_batches(wins, bs, shuffle=True):
    """Both paths of the backbone are causal (masked attention + a unidirectional GRU) and the head packs
    sequences, so right-padding never reaches a real token: plain chunks are fine."""
    wins = list(wins)
    if shuffle:
        random.shuffle(wins)
    return [wins[i:i + bs] for i in range(0, len(wins), bs)]

## 5. Model head, adaptation methods, distillation student

In [ ]:
class LoRALinear(nn.Module):
    """Plain LoRA: W x + (alpha/r) * B A x, with W frozen. (Not LORMA - see the note at the top.)"""
    def __init__(self, base, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.T @ self.B.T) * self.scale

def apply_lora(module, r=16, alpha=32):
    """Wraps every nn.Linear under `module`. In the fusion blocks that is: attn.qkv, attn.out, mlp.fc1, mlp.fc2,
    and (in the GRU blocks) gate and rnn.proj. The GRU's own recurrent weights are not nn.Linear, so they stay frozen."""
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRALinear(child, r, alpha))
        else:
            apply_lora(child, r, alpha)

def n_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

In [ ]:
class NERModel(nn.Module):
    """Phase-1 fusion LM (hidden states, not logits) + BiGRU + linear tag head.
    The backbone is causal in both paths, so a token cannot see its right-hand context; the BiGRU restores it."""
    def __init__(self, lm, rnn_hidden=256, dropout=0.1):
        super().__init__()
        self.lm = lm
        self.rnn = nn.GRU(lm.cfg.d_model, rnn_hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * rnn_hidden, N_TAGS)

    def head_params(self):
        return list(self.rnn.parameters()) + list(self.out.parameters())

    def features(self, x):
        # same path as FusionLM.forward, stopping before lm_head. RoPE tables replace the additive position embedding.
        lm, T = self.lm, x.size(1)
        h = lm.drop(lm.tok_emb(x))
        cos, sin = lm.rope_cos[:T], lm.rope_sin[:T]
        for b in lm.blocks:
            h = b(h, cos, sin)
        return lm.norm_f(h)

    def forward(self, x, lens):
        h = self.features(x)
        packed = nn.utils.rnn.pack_padded_sequence(h, lens.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.rnn(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=x.size(1))
        return self.out(self.drop(h))


def build_model(method):
    lm = load_lm()
    if method == "lora":
        for p in lm.parameters(): p.requires_grad = False
        apply_lora(lm.blocks)
    elif method == "partial":                       # freeze embeddings + lower half of the blocks
        for p in lm.tok_emb.parameters(): p.requires_grad = False
        for b in lm.blocks[: len(lm.blocks) // 2]:
            for p in b.parameters(): p.requires_grad = False
    elif method == "bitfit":                        # this model has almost no biases (only the gate's), so: biases + norm scales
        for n, p in lm.named_parameters(): p.requires_grad = ("bias" in n) or ("norm" in n)
    elif method != "full":
        raise ValueError(method)
    return NERModel(lm).to(dev)


def make_student(teacher, keep=None):
    """Layer-pruned student: alternate PAIRS of blocks of the teacher (default) + shared embeddings, fresh head.
    Pairs, not single blocks, because the GRU branch sits in blocks with i % rnn_every == 0: keeping blocks
    0,1,4,5,8,9 of a 10-block teacher gives a 6-block student with exactly the same GRU / no-GRU pattern, so every
    kept block loads cleanly. (Keeping every 2nd block would give an all-GRU student.)"""
    t = teacher.lm
    cfg = copy.copy(t.cfg)
    k = max(cfg.rnn_every, 1)
    keep = keep or [i for i in range(cfg.n_layers) if (i // k) % 2 == 0]
    cfg.n_layers = len(keep)
    lm = FusionLM(cfg)
    for i, j in enumerate(keep):
        assert (lm.blocks[i].rnn is None) == (t.blocks[j].rnn is None), \
            f"student block {i} / teacher block {j} differ in GRU presence; choose `keep` so the patterns match"
        lm.blocks[i].load_state_dict(t.blocks[j].state_dict())
    lm.tok_emb.load_state_dict(t.tok_emb.state_dict())          # also updates the tied lm_head
    lm.norm_f.load_state_dict(t.norm_f.state_dict())
    print(f"student keeps teacher blocks {keep}")
    return NERModel(lm).to(dev)

### 5.1 Check that the re-implemented forward path matches the Phase-1 model
`NERModel.features` re-states `FusionLM.forward` minus the LM head. This check makes sure the two agree, and shows that the backbone really is causal (which is why the head needs a BiGRU).

In [ ]:
_lm = load_lm().eval()
_x = torch.randint(0, tokenizer.vocab_size, (2, min(100, _lm.cfg.max_seq_len)))
_ner = NERModel(_lm).eval()
with torch.no_grad():
    _a = _lm(_x)                                     # Phase-1 logits
    _b = _lm.lm_head(_ner.features(_x))              # logits rebuilt from NERModel.features
    _t = _x.size(1) // 2
    _x2 = _x.clone(); _x2[:, _t + 1:] = torch.randint(0, tokenizer.vocab_size, (2, _x.size(1) - _t - 1))
    _leak = (_ner.features(_x)[:, :_t + 1] - _ner.features(_x2)[:, :_t + 1]).abs().max().item()
print(f"max |logits - lm_head(features)| = {(_a - _b).abs().max().item():.2e}  (must be ~0)")
print(f"backbone change on the unchanged prefix when the future is altered = {_leak:.2e}  (~0: causal, so the BiGRU head is needed)")
assert torch.allclose(_a, _b, atol=1e-4)
assert _leak < 1e-4
del _lm, _ner, _x, _x2, _a, _b

## 6. Inference and evaluation (overlapping windows, averaged log-probs)

In [ ]:
@torch.no_grad()
def predict_docs(model, docs, labels, win=WIN, stride=STRIDE, bs=16):
    """Overlapping windows; per-token log-probs are averaged over every window that covers the token."""
    model.eval()
    out = {}
    for d in docs:
        n = len(d["ids"])
        if n == 0:
            out[d["id"]] = []; continue
        starts = win_starts(n, win, stride)           # all windows have equal length (or the doc is one short window)
        acc, cnt = torch.zeros(n, N_TAGS), torch.zeros(n, 1)
        for i in range(0, len(starts), bs):
            ss = starts[i:i + bs]
            X, _, lens = collate([(d["ids"][s:s + win], None) for s in ss])
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                lg = model(X.to(dev), lens)
            lp = F.log_softmax(lg.float(), -1).cpu()
            for j, s in enumerate(ss):
                L = int(lens[j]); acc[s:s + L] += lp[j, :L]; cnt[s:s + L] += 1
        out[d["id"]] = decode((acc / cnt).argmax(-1).tolist(), d["offs"], d["text"], labels)
    return out

def evaluate(model, docs, labels):
    pred = predict_docs(model, docs, labels)
    return strict_scores(pred, {d["id"]: d["ents"] for d in docs}, labels)

## 7. Training loop (shared by all methods; best-epoch restore on the validation fold)

In [ ]:
def train(method, train_docs, val_docs, labels, epochs=8, bs=4, accum=4,
          lr_bb=5e-5, lr_head=1e-3, teacher=None, kd_T=2.0, kd_alpha=0.5, o_weight=0.3,
          win=WIN, stride=STRIDE, seed=0):
    """method: full | lora | partial | bitfit | kd.   Returns (model restored to its best-val epoch, best val F1)."""
    random.seed(seed); torch.manual_seed(seed)
    if method == "kd":
        assert teacher is not None, "pass teacher=<your fully fine-tuned model>"
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
        model = make_student(teacher)
    else:
        model = build_model(method)
    print(f"[{method}] params total {n_params(model)/1e6:.1f}M | trainable {n_params(model, True)/1e6:.2f}M")

    head_ids = {id(p) for p in model.head_params()}
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    hd = [p for p in model.parameters() if id(p) in head_ids]
    opt = torch.optim.AdamW([{"params": bb, "lr": lr_bb}, {"params": hd, "lr": lr_head}], weight_decay=0.01)
    base_lrs = [lr_bb, lr_head]

    wins = make_windows(train_docs, win, stride)
    steps_total = epochs * math.ceil(len(iter_batches(wins, bs, False)) / accum)
    warm = max(1, int(0.06 * steps_total))
    def lr_mult(s):
        return s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps_total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    cw = torch.ones(N_TAGS, device=dev); cw[0] = o_weight         # down-weight the dominant O class

    best, best_state, step = -1.0, None, 0
    for ep in range(epochs):
        model.train(); t0 = time.time(); run = 0.0
        batches = iter_batches(wins, bs, True)
        for bi, batch in enumerate(batches):
            X, Y, lens = collate(batch); X, Y = X.to(dev), Y.to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                lg = model(X, lens)
            loss = F.cross_entropy(lg.float().reshape(-1, N_TAGS), Y.reshape(-1), weight=cw, ignore_index=-100)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                    tl = teacher(X, lens)
                m = Y != -100
                kl = F.kl_div(F.log_softmax(lg.float()[m] / kd_T, -1), F.softmax(tl.float()[m] / kd_T, -1),
                              reduction="batchmean") * kd_T ** 2
                loss = kd_alpha * loss + (1 - kd_alpha) * kl
            scaler.scale(loss / accum).backward(); run += loss.item()
            if (bi + 1) % accum == 0 or bi + 1 == len(batches):
                step += 1
                for g, b in zip(opt.param_groups, base_lrs): g["lr"] = b * lr_mult(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        msg = f"ep {ep+1}/{epochs} loss {run/max(1, len(batches)):.4f} ({time.time()-t0:.0f}s)"
        if val_docs:
            sc = evaluate(model, val_docs, labels)
            msg += f" | val strict macro-F1 {sc['macro_f1']:.4f} (P {sc['macro_p']:.4f} R {sc['macro_r']:.4f})"
            if sc["macro_f1"] > best:
                best = sc["macro_f1"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(msg)
    if best_state is not None:
        model.load_state_dict(best_state); print(f"restored best epoch, val macro-F1 {best:.4f}")
    return model, best

## 8. Submission writer
`prediction.json` goes at the **root** of the zip. Upload each zip to Codabench.

In [ ]:
def make_submission(model, test_docs, labels, tag):
    """Writes prediction.json and wraps it as submission_p2_<tag>.zip with prediction.json at the zip root."""
    pred = predict_docs(model, test_docs, labels)
    assert set(pred) == {d["id"] for d in test_docs}, "missing / unknown test ids"
    sub = {did: [[s, e, l] for s, e, l in es] for did, es in pred.items()}
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(sub, f, ensure_ascii=False)
    name = f"{OUT}/submission_p2_{tag}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print(name, zipfile.ZipFile(name).namelist(), "| docs:", len(sub), "| entities:", sum(len(v) for v in sub.values()))
    return name

results = {}                                    # method -> best val strict macro-F1

# Starting hyper-parameters per method (carried over from the decoder run - re-tune them for the fusion model).
# Kept in one place so the final fold1+fold2 retrain in Section 14 reuses exactly what the experiment used.
HP = {
    "full":    dict(epochs=8,  lr_bb=5e-5, lr_head=1e-3),
    "lora":    dict(epochs=15, lr_bb=2e-3, lr_head=2.5e-3),
    "partial": dict(epochs=15, lr_bb=1e-4, lr_head=2e-3),
    "kd":      dict(epochs=8,  lr_bb=1e-4, lr_head=1e-3),
}

## 9. Experiment 1 - full fine-tuning
Everything trainable (backbone lr 5e-5, head lr 1e-3). This model is also the **teacher** for the distillation run, so keep it in memory. Train on fold1, validate on fold2 (best-epoch restore).

In [ ]:
m_full, results["full"] = train("full", docs1, docs2, labels, **HP["full"])
torch.save(m_full.state_dict(), f"{OUT}/p2_fusion_full.pt")        # insurance against a session crash
make_submission(m_full, test_docs, labels, "fusion_full")

## 10. Experiment 2 - LoRA
Backbone frozen, low-rank adapters on every `nn.Linear` in the blocks (attention `qkv`/`out`, MLP, GRU gate and GRU output projection). The backbone GRU's recurrent weights are *not* adapted, only frozen. LoRA wants a much larger learning rate than full fine-tuning.

In [ ]:
m_lora, results["lora"] = train("lora", docs1, docs2, labels, **HP["lora"])
make_submission(m_lora, test_docs, labels, "fusion_lora")

## 11. Experiment 3 - partial (layer-wise) fine-tuning
Embeddings and the lower half of the blocks frozen; only the upper half + head train (for a 10-block model that is blocks 5-9, which contain the GRU branches of blocks 6 and 8). Swap `"partial"` for `"bitfit"` if you prefer a bias-only method (here: biases + norm scales).

In [ ]:
m_part, results["partial"] = train("partial", docs1, docs2, labels, **HP["partial"])
make_submission(m_part, test_docs, labels, "fusion_partial")

## 12. Experiment 4 - knowledge distillation
Teacher = experiment 1 (full fine-tune). Student = the teacher with alternate pairs of blocks removed (layer-pruned initialisation that keeps the GRU / no-GRU pattern), trained on hard labels + temperature-scaled teacher logits. Compare its parameter count and F1 to the teacher in your report.

In [ ]:
m_kd, results["kd"] = train("kd", docs1, docs2, labels, teacher=m_full, **HP["kd"])
make_submission(m_kd, test_docs, labels, "fusion_kd")

## 13. Summary

In [ ]:
print("val strict macro-F1 (train on fold1, validate on fold2):")
for k, v in results.items():
    print(f"  {k:8s} {v:.4f}")
print("\nper-label F1 of the full model (error analysis starter):")
sc = evaluate(m_full, docs2, labels)
for k, v in sorted(sc["per_label"].items(), key=lambda kv: kv[1]):
    print(f"  {k:10s} {v:.3f}")

## 14. (Optional) retrain on fold1 + fold2
Once you know the best method and epoch count, retrain on all labelled data (no validation set, so no best-epoch restore) and submit again. Set `FINAL_METHOD` to the winner (`full`, `lora` or `partial`; `kd` needs a teacher, so run it by hand) and `BEST_EPOCHS` to the epoch with the best val F1 in the fold1-only run (read it from the printed `ep ...` lines). The learning rates come from `HP`.

`doc_logprobs` / `save_logprobs` also dump the per-token log-probs, for ensembling with other models' outputs. The fusion model uses the same tokenizer as the other Phase-1 models, so the token offsets line up.

In [ ]:
@torch.no_grad()
def doc_logprobs(model, docs, win=WIN, stride=STRIDE, bs=16):
    """Per-token log-probs (n_tokens, N_TAGS), averaged over overlapping windows. Same logic as predict_docs."""
    model.eval()
    out = {}
    for d in docs:
        n = len(d["ids"])
        if n == 0:
            out[d["id"]] = torch.zeros(0, N_TAGS); continue
        starts = win_starts(n, win, stride)
        acc, cnt = torch.zeros(n, N_TAGS), torch.zeros(n, 1)
        for i in range(0, len(starts), bs):
            ss = starts[i:i + bs]
            X, _, lens = collate([(d["ids"][s:s + win], None) for s in ss])
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                lg = model(X.to(dev), lens)
            lp = F.log_softmax(lg.float(), -1).cpu()
            for j, s in enumerate(ss):
                L = int(lens[j]); acc[s:s + L] += lp[j, :L]; cnt[s:s + L] += 1
        out[d["id"]] = (acc / cnt).half()
    return out

def save_logprobs(model, docs, tag, with_gold=False):
    pack = {"lp": doc_logprobs(model, docs), "offs": {d["id"]: d["offs"] for d in docs},
            "text": {d["id"]: d["text"] for d in docs}, "labels": labels}
    if with_gold:
        pack["gold"] = {d["id"]: d["ents"] for d in docs}
    torch.save(pack, f"{OUT}/lp_{tag}.pt")
    print("saved", f"{OUT}/lp_{tag}.pt")

In [ ]:
FINAL_METHOD = "full"   # <- the method that won on val (full | lora | partial)
BEST_EPOCHS = 8         # <- the epoch with the best val F1 in your fold1-only run
assert FINAL_METHOD != "kd", "kd needs a teacher; call train('kd', ..., teacher=...) yourself"
for name in ["m_full", "m_lora", "m_part", "m_kd"]:     # free GPU memory first
    globals().pop(name, None)
import gc; gc.collect(); torch.cuda.empty_cache()

m_final, _ = train(FINAL_METHOD, docs1 + docs2, None, labels, seed=1, **{**HP[FINAL_METHOD], "epochs": BEST_EPOCHS})
make_submission(m_final, test_docs, labels, "fusion_final")   # also a valid submission on its own
save_logprobs(m_final, test_docs, "fusion_test")              # for ensembling with the decoder / other models